In [68]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, f1_score
import joblib
import json
from datetime import datetime
import os
from sklearn.preprocessing import LabelEncoder

# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

# Load datasets
meteorological = pd.read_csv("Meteorological_data.csv")
controls = pd.read_csv("Malaria_control_measures.csv")
patient = pd.read_csv("Malaria_surveillance_data.csv")

In [69]:
# Fill missing values with district-week medians
def fill_missing(df, group_cols, fill_cols):
    for col in fill_cols:
        df[col] = df.groupby(group_cols)[col].transform(lambda x: x.fillna(x.median()))
    return df

meteorological = fill_missing(
    meteorological,
    group_cols=['District', 'Week'],
    fill_cols=['Rainfall', 'Min Temperature', 'Max Temperature', 'Humidity']
)

controls = fill_missing(
    controls,
    group_cols=['District', 'Week'],
    fill_cols=['Effective_Net_Usage', 'IRS_Coverage']
)

patient = fill_missing(
    patient,
    group_cols=['District', 'Week'],
    fill_cols=['average_latitude', 'average_longitude', 'Month']
)


In [70]:
def calculate_outbreak(df):
    df = df.sort_values(['District', 'Year', 'Week'])
    
    # Use EXPANDING window (only past data)
    df['hist_avg'] = df.groupby('District')['true_malaria_cases'].transform(
        lambda x: x.expanding(min_periods=4).mean().shift(1))
    df['hist_std'] = df.groupby('District')['true_malaria_cases'].transform(
        lambda x: x.expanding(min_periods=4).std().shift(1))
    df['threshold'] = df['hist_avg'] + (2 * df['hist_std'])
    df['malaria_outbreak'] = (df['true_malaria_cases'] > df['threshold']).astype(int)
    print(df['malaria_outbreak'].value_counts())
    return df

patient = calculate_outbreak(patient)


malaria_outbreak
0    3201
1     231
Name: count, dtype: int64


In [71]:
print(meteorological.columns)
# Approximate month from week
meteorological['Month'] = ((meteorological['Week'] - 1) // 4 + 1).clip(upper=12)


Index(['District', 'Year', 'Week', 'Humidity', 'Rainfall', 'Min Temperature',
       'Max Temperature'],
      dtype='object')


In [72]:
def create_features(df):
    # Cyclical time features
    df['month_sin'] = np.sin(2 * np.pi * df['Month']/12)
    df['month_cos'] = np.cos(2 * np.pi * df['Month']/12)
    
    # Environmental interactions
    df['temp_range'] = df['Max Temperature'] - df['Min Temperature']
    df['temp_humidity'] = df['Max Temperature'] * df['Humidity']
    
    return df

meteorological = create_features(meteorological)

In [73]:
def add_lags(df, features, lags=[1, 2, 3, 4]):
    df = df.sort_values(['District', 'Year', 'Week'])
    for feature in features:
        for lag in lags:
            df[f'{feature}_lag_{lag}'] = df.groupby('District')[feature].shift(lag)
    return df

patient = add_lags(patient, ['true_malaria_cases'])

In [74]:
patient.head()

,District,Year,Week,true_malaria_cases,prop_male,prop_female,prop_<5_age,prop_5_14_age,prop_15_29_age,prop_30_44_age,...,occ_business_owner,Month,hist_avg,hist_std,threshold,malaria_outbreak,true_malaria_cases_lag_1,true_malaria_cases_lag_2,true_malaria_cases_lag_3,true_malaria_cases_lag_4
2496,CHAKE-CHAKE,2018,1,7,0.571429,0.428571,0.142857,0.571429,0.285714,0.000000,...,0,1.0,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
2497,CHAKE-CHAKE,2018,2,6,0.333333,0.500000,0.333333,0.000000,0.333333,0.166667,...,0,1.0,NaN,NaN,NaN,0,7.0,NaN,NaN,NaN
2498,CHAKE-CHAKE,2018,3,5,0.600000,0.400000,0.000000,0.400000,0.400000,0.200000,...,0,1.0,NaN,NaN,NaN,0,6.0,7.0,NaN,NaN
2499,CHAKE-CHAKE,2018,4,2,1.000000,0.000000,0.000000,0.500000,0.500000,0.000000,...,0,1.0,NaN,NaN,NaN,0,5.0,6.0,7.0,NaN
2500,CHAKE-CHAKE,2018,5,3,1.000000,0.000000,0.000000,0.666667,0.333333,0.000000,...,0,1.0,5.0,2.160247,9.320494,0,2.0,5.0,6.0,7.0


In [75]:
def clean_data(meteorological, controls, patient):
    # 1. Meteorological Data
    meteorological = fill_missing(meteorological, ['District', 'Month'], 
                               ['Humidity', 'Rainfall', 'Min Temperature', 'Max Temperature'])
    meteorological['Rainfall'] = meteorological['Rainfall'].fillna(0)
    
    # 2. Control Measures
    controls['IRS_Coverage'] = controls.groupby('District')['IRS_Coverage'].transform(
        lambda x: x.fillna(method='ffill').fillna(method='bfill').fillna(0))
    
    # 3. Patient Data
    patient = calculate_outbreak(patient)
    
    # 4. Merge all
    combined = pd.merge(patient, meteorological, on=['District', 'Year', 'Week', "Month"], how='left')
    combined = pd.merge(combined, controls, on=['District', 'Year', 'Week'], how='left')
    
    # 5. Handle lags
    for lag in [1, 2, 3, 4]:
        combined[f'true_malaria_cases_lag_{lag}'] = combined.groupby('District')['true_malaria_cases'].shift(lag)
        combined[f'true_malaria_cases_lag_{lag}'].fillna(0, inplace=True)
    
    # 6. Final null check
    combined = combined.fillna(combined.median(numeric_only=True))
    combined = combined.fillna(0)
    
    return combined
print(patient.info())
print(patient['District'].unique())
combined = clean_data(meteorological, controls, patient)
combined.isna().sum()
le = LabelEncoder()
combined['District'] = le.fit_transform(combined['District'])
# Create 'Date' column from 'Year' and 'Week'
combined['Date'] = pd.to_datetime(combined['Year'].astype(str) + combined['Week'].astype(str).str.zfill(2) + '1', format='%G%V%u')
# Time-based split (80-20)
split_date = combined['Date'].quantile(0.8)
train = combined[combined['Date'] < split_date].copy()
test = combined[combined['Date'] >= split_date].copy()


<class 'pandas.core.frame.DataFrame'>
Index: 3432 entries, 2496 to 1247
Data columns (total 36 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   District                  3432 non-null   object 
 1   Year                      3432 non-null   int64  
 2   Week                      3432 non-null   int64  
 3   true_malaria_cases        3432 non-null   int64  
 4   prop_male                 3432 non-null   float64
 5   prop_female               3432 non-null   float64
 6   prop_<5_age               3432 non-null   float64
 7   prop_5_14_age             3432 non-null   float64
 8   prop_15_29_age            3432 non-null   float64
 9   prop_30_44_age            3432 non-null   float64
 10  prop_45+_age              3432 non-null   float64
 11  is_index                  3432 non-null   int64  
 12  is_additional_index       3432 non-null   int64  
 13  is_other_member           3432 non-null   int64  
 14  is_private

In [76]:
combined.head()
print(combined.info())
# print(combined.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3432 entries, 0 to 3431
Data columns (total 47 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   District                  3432 non-null   int64         
 1   Year                      3432 non-null   int64         
 2   Week                      3432 non-null   int64         
 3   true_malaria_cases        3432 non-null   int64         
 4   prop_male                 3432 non-null   float64       
 5   prop_female               3432 non-null   float64       
 6   prop_<5_age               3432 non-null   float64       
 7   prop_5_14_age             3432 non-null   float64       
 8   prop_15_29_age            3432 non-null   float64       
 9   prop_30_44_age            3432 non-null   float64       
 10  prop_45+_age              3432 non-null   float64       
 11  is_index                  3432 non-null   int64         
 12  is_additional_index 

In [77]:
print(combined.head())

   District  Year  Week  true_malaria_cases  prop_male  prop_female  \
0         0  2018     1                   7   0.571429     0.428571   
1         0  2018     2                   6   0.333333     0.500000   
2         0  2018     3                   5   0.600000     0.400000   
3         0  2018     4                   2   1.000000     0.000000   
4         0  2018     5                   3   1.000000     0.000000   

   prop_<5_age  prop_5_14_age  prop_15_29_age  prop_30_44_age  ...  Rainfall  \
0     0.142857       0.571429        0.285714        0.000000  ...      14.1   
1     0.333333       0.000000        0.333333        0.166667  ...       1.1   
2     0.000000       0.400000        0.400000        0.200000  ...       0.0   
3     0.000000       0.500000        0.500000        0.000000  ...       0.0   
4     0.000000       0.666667        0.333333        0.000000  ...       7.0   

   Min Temperature  Max Temperature  month_sin     month_cos  temp_range  \
0        24.3428

In [78]:
# Selected features (adjust based on your EDA)
features = [
    # location
    "District",
    "average_latitude",
    "average_longitude",

    #time
    "Year",
    "Month",
    "Week",

    #personal infos
    # #gender
    # "prop_male",
    # "prop_female",

    # #age
    # "prop_<5_age",
    # "prop_5_14_age",
    # "prop_15_29_age",
    # "prop_30_44_age",
    # "prop_45+_age",

    # #member type
    # "is_index",
    # "is_additional_index",
    # "is_other_member",

    # #facility type
    # "is_private",
    # "is_public",

    # #occupation
    # "occ_student",
    # "occ_health_worker",
    # "occ_police_officer",
    # "occ_watchman",
    # "occ_farmer",
    # "occ_fisherman",
    # "occ_unemployed",
    # "occ_housewife",
    # "occ_business_owner",
    
    # "true_malaria_cases_lag_1",
    # "true_malaria_cases_lag_2",
    # "true_malaria_cases_lag_3",
    # "true_malaria_cases_lag_4",

    #environmental conditions
    "Humidity",
    "Rainfall",
    "Min Temperature",
    "Max Temperature",
    "temp_range",
    # "temp_humidity",

    #controls
    "Effective_Net_Usage",
    "IRS_Coverage",
]


X_train = train[features].fillna(train[features].median())
y_train = train['malaria_outbreak']
X_test = test[features].fillna(test[features].median())
y_test = test['malaria_outbreak']

In [79]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

In [80]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)

model.fit(X_train_res, y_train_res)

RandomForestClassifier(class_weight='balanced', max_depth=10, n_estimators=200,
                       random_state=42)

In [81]:
# Predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Metrics
print("Test Performance:")
print(classification_report(y_test, y_pred))
print(f"ROC AUC: {roc_auc_score(y_test, y_proba):.3f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.3f}")

Test Performance:
              precision    recall  f1-score   support

           0       0.91      1.00      0.95       633
           1       0.00      0.00      0.00        60

    accuracy                           0.91       693
   macro avg       0.46      0.50      0.48       693
weighted avg       0.83      0.91      0.87       693

ROC AUC: 0.720
F1 Score: 0.000


In [83]:
# Save model & artifacts
MODEL_DIR = "../malaria_prediction_system/"
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. Save model
joblib.dump(model, f"{MODEL_DIR}model.pkl")

# 2. Save scaler (if used)
scaler = StandardScaler()
scaler.fit(X_train_res)
joblib.dump(scaler, f"{MODEL_DIR}scaler.pkl")
joblib.dump(le, f"{MODEL_DIR}encoder.pkl")
# 3. Save feature list
with open(f"{MODEL_DIR}features.json", "w") as f:
    json.dump(features, f)

# 4. Save metadata
metadata = {
    "train_date_range": {
        "start": train['Date'].min().strftime("%Y-%m-%d"),
        "end": train['Date'].max().strftime("%Y-%m-%d")
    },
    "model_type": "RandomForestClassifier",
    "version": "1.0.0",
    "required_features": features
}

with open(f"{MODEL_DIR}metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)